In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv("D:/vs code/telecom-revenue-churn-analytics/data/Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [11]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [12]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [13]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [18]:

# ------------------------------------------------
# 1. DROP CUSTOMER ID
# ------------------------------------------------
# customerID is a unique identifier.
# It does not carry any behavioral or business signal.
# Keeping it can mislead ML models and reduces interpretability.
df.drop(columns=["customerID"], inplace=True)



KeyError: "['customerID'] not found in axis"

In [17]:
# ==============================
# FEATURE ENGINEERING
# ==============================
# Goal:
# Convert raw telecom data into business-meaningful features
# that help analyze churn, revenue impact, and customer behavior
# ==============================



# ------------------------------------------------
# 1. ARPU (Average Revenue Per User)
# ------------------------------------------------
# In telecom, ARPU is a key KPI used by business teams.
# MonthlyCharges already represents monthly revenue per customer,
# so we expose it explicitly as ARPU for clarity in analysis and dashboards.
df["ARPU"] = df["MonthlyCharges"]


# ------------------------------------------------
# 2. TENURE GROUPING
# ------------------------------------------------
# Raw tenure (in months) is hard to interpret in dashboards.
# Telecom teams usually analyze churn by tenure buckets
# because churn risk is highest in early customer lifecycle.

def tenure_group(tenure):
    """
    Categorize customers based on how long they have stayed with the company.
    This helps in identifying churn-prone lifecycle stages.
    """
    if tenure <= 12:
        return "0-1 year"
    elif tenure <= 24:
        return "1-2 years"
    elif tenure <= 48:
        return "2-4 years"
    else:
        return "4+ years"

df["TenureGroup"] = df["tenure"].apply(tenure_group)


# ------------------------------------------------
# 3. REVENUE LOST DUE TO CHURN
# ------------------------------------------------
# Business question:
# Not just "how many customers churned?",
# but "how much revenue was lost due to churn?"
#
# If a customer churned, we consider their monthly revenue
# as revenue lost for the next billing cycle.
df["RevenueLost"] = np.where(
    df["Churn"] == "Yes",
    df["MonthlyCharges"],
    0
)


# ------------------------------------------------
# 4. BINARY TARGET VARIABLE FOR ML
# ------------------------------------------------
# Machine learning models require numerical targets.
# Convert Churn (Yes/No) into binary format.
df["ChurnFlag"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})


# ------------------------------------------------
# 5. FINAL CHECK
# ------------------------------------------------
# Preview engineered dataset
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ARPU,TenureGroup,RevenueLost,ChurnFlag
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,29.85,0-1 year,0.00,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,One year,No,Mailed check,56.95,1889.50,No,56.95,2-4 years,0.00,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,53.85,0-1 year,53.85,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,42.30,2-4 years,0.00,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,70.70,0-1 year,70.70,1


In [19]:
df.to_csv("../data/telco_cleaned.csv", index=False)

STEP 2: Exploratory Data Analysis (EDA)

Objective

Understand why customers churn

Identify revenue impact

Generate clear KPIs that will later go into Power BI

In [20]:
# Shape of dataset
df.shape

(7043, 24)

In [21]:
# Data types and null check
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [22]:
# Statistical summary
df.describe(include="all")

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ARPU,TenureGroup,RevenueLost,ChurnFlag
count,7043,7043.000000,7043,7043,7043.000000,7043,7043,7043,7043,7043,...,7043,7043,7043,7043.000000,7043.000000,7043,7043.000000,7043,7043.000000,7043.000000
unique,2,NaN,2,2,NaN,2,3,3,3,3,...,3,2,4,NaN,NaN,2,NaN,4,NaN,NaN
top,Male,NaN,No,No,NaN,Yes,No,Fiber optic,No,No,...,Month-to-month,Yes,Electronic check,NaN,NaN,No,NaN,4+ years,NaN,NaN
freq,3555,NaN,3641,4933,NaN,6361,3390,3096,3498,3088,...,3875,4171,2365,NaN,NaN,5174,NaN,2239,NaN,NaN
mean,NaN,0.162147,NaN,NaN,32.371149,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,64.761692,2279.734304,NaN,64.761692,NaN,19.754487,0.265370
std,NaN,0.368612,NaN,NaN,24.559481,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,30.090047,2266.794470,NaN,30.090047,NaN,35.239967,0.441561
min,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,18.250000,0.000000,NaN,18.250000,NaN,0.000000,0.000000
25%,NaN,0.000000,NaN,NaN,9.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,35.500000,398.550000,NaN,35.500000,NaN,0.000000,0.000000
50%,NaN,0.000000,NaN,NaN,29.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,70.350000,1394.550000,NaN,70.350000,NaN,0.000000,0.000000
75%,NaN,0.000000,NaN,NaN,55.000000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,89.850000,3786.600000,NaN,89.850000,NaN,24.100000,1.000000


2.2 Churn Distribution (Core Telecom KPI)

In [23]:
# Churn percentage
churn_dist = df["Churn"].value_counts(normalize=True) * 100
churn_dist

Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64

In [24]:
# Absolute churn count
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

2.3 Revenue Impact of Churn


In [25]:
# Total monthly revenue
total_revenue = df["MonthlyCharges"].sum()

# Revenue lost due to churn
revenue_lost = df["RevenueLost"].sum()

total_revenue, revenue_lost

(np.float64(456116.6), np.float64(139130.84999999998))

In [26]:
# % revenue loss
(revenue_lost / total_revenue) * 100

np.float64(30.503351555282133)

In [27]:
# Churn rate by tenure group
churn_by_tenure = (
    df.groupby("TenureGroup")["ChurnFlag"]
      .mean()
      .sort_values(ascending=False)
)

churn_by_tenure

TenureGroup
0-1 year     0.474382
1-2 years    0.287109
2-4 years    0.203890
4+ years     0.095132
Name: ChurnFlag, dtype: float64

In [ ]:
# Average ARPU by churn
df.groupby("Churn")["ARPU"].mean()

In [ ]:
# Churn rate by contract type
(
    df.groupby("Contract")["ChurnFlag"]
      .mean()
      .sort_values(ascending=False)
)